In [2]:
"""
Presentation creator: outline chat -> slide-by-slide editing -> .pptx export.

Setup (PowerShell):
    pip install gradio ollama pydantic python-pptx
    ollama pull qwen3:8b        # or change MODEL below
    python app.py
"""
import math
import pathlib
import re
from typing import Literal

import gradio as gr
import ollama
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE
from pptx.util import Inches, Pt
from pydantic import BaseModel, Field

MODEL = "qwen3.5"
SAVE_PATH = pathlib.Path("deck.json")
EXPORT_PATH = pathlib.Path("deck.pptx")
LAYOUTS = ["title", "bullets", "two_column", "image_left", "quote"]
LayoutName = Literal["title", "bullets", "two_column", "image_left", "quote"]


# ----------------------------------------------------------------------------
# Data model: the deck is the single source of truth
# ----------------------------------------------------------------------------
class SlideDraft(BaseModel):
    """What the LLM returns for one slide."""
    title: str
    bullets: list[str]
    layout: LayoutName
    notes: str


class OutlineReply(BaseModel):
    """What the LLM returns in the outline phase."""
    reply: str
    outline: list[str]


class Slide(BaseModel):
    intent: str = ""  # the outline line this slide came from
    title: str = ""
    bullets: list[str] = Field(default_factory=list)
    layout: LayoutName = "bullets"
    notes: str = ""
    status: Literal["pending", "draft", "approved"] = "pending"
    chat: list[dict[str, str]] = Field(default_factory=list)


class Deck(BaseModel):
    topic: str = ""
    outline: list[str] = Field(default_factory=list)
    outline_locked: bool = False
    outline_chat: list[dict[str, str]] = Field(default_factory=list)
    slides: list[Slide] = Field(default_factory=list)
    current: int = 0


def save(deck: Deck) -> None:
    SAVE_PATH.write_text(deck.model_dump_json(indent=2), encoding="utf-8")


# ----------------------------------------------------------------------------
# LLM helpers
# ----------------------------------------------------------------------------
def llm(prompt: str, model_cls):
    try:
        r = ollama.chat(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            format=model_cls.model_json_schema(),
        )
        return model_cls.model_validate_json(r.message.content)
    except Exception as e:  # surface in the UI instead of crashing
        raise gr.Error(f"LLM call failed: {e}")


def parse_lines(text: str) -> list[str]:
    out = []
    for line in (text or "").splitlines():
        line = re.sub(r"^\s*(\d+[.)]|[-*•])\s*", "", line).strip()
        if line:
            out.append(line)
    return out


def numbered(items: list[str]) -> str:
    return "\n".join(f"{i + 1}. {t}" for i, t in enumerate(items)) or "(empty)"


# ----------------------------------------------------------------------------
# Phase 1: outline
# ----------------------------------------------------------------------------
def outline_chat(msg, deck: Deck, outline_text):
    msg = (msg or "").strip()
    if not msg:
        return deck, deck.outline_chat, "", outline_text
    deck.outline = parse_lines(outline_text)  # user's manual edits win
    if not deck.topic:
        deck.topic = msg
    convo = "\n".join(f"{m['role']}: {m['content']}" for m in deck.outline_chat[-8:])
    prompt = f"""You help a user design a presentation outline.
Topic / goal (from the first message): {deck.topic}
Current outline (the user may have edited it by hand; respect their edits):
{numbered(deck.outline)}

Recent conversation:
{convo}

User: {msg}

Return JSON with:
- reply: a short, friendly message (1-3 sentences) about what you changed or a question for the user
- outline: the FULL updated outline as a list of slide titles, one short line per slide, in order.
  Ask for the outline you think best fits; 6-12 slides unless the user says otherwise."""
    res = llm(prompt, OutlineReply)
    deck.outline = [t.strip() for t in res.outline if t.strip()]
    deck.outline_chat.append({"role": "user", "content": msg})
    deck.outline_chat.append({"role": "assistant", "content": res.reply})
    save(deck)
    return deck, deck.outline_chat, "", "\n".join(deck.outline)


def approve_outline(deck: Deck, outline_text):
    items = parse_lines(outline_text)
    if not items:
        raise gr.Error("The outline is empty.")
    old = {s.intent: s for s in deck.slides}  # keep work on slides whose line is unchanged
    deck.outline = items
    deck.slides = [old.get(t) or Slide(intent=t, title=t) for t in items]
    deck.outline_locked = True
    deck.current = 0
    save(deck)
    return render(deck, "slides")


# ----------------------------------------------------------------------------
# Phase 2: slide loop
# ----------------------------------------------------------------------------
def render(deck: Deck, selected: str | None = None):
    """Return values for OUT (see wiring below)."""
    if deck.slides:
        s = deck.slides[deck.current]
        progress = (f"**Slide {deck.current + 1} / {len(deck.slides)}**  ·  "
                    f"_{s.intent}_  ·  status: **{s.status}**")
    else:
        s, progress = Slide(), ""
    all_done = bool(deck.slides) and all(x.status == "approved" for x in deck.slides)
    return (
        deck,
        gr.Tabs(selected=selected) if selected else gr.update(),
        gr.Tab(interactive=deck.outline_locked),
        gr.Tab(interactive=all_done),
        progress,
        s.title,
        "\n".join(s.bullets),
        s.layout,
        s.notes,
        s.chat,
        "",
    )


def read_fields(deck: Deck, title, bullets, layout, notes) -> Deck:
    """Copy what the user typed in the form into the current slide."""
    s = deck.slides[deck.current]
    s.title = (title or "").strip()
    s.bullets = parse_lines(bullets)
    s.layout = layout
    s.notes = notes or ""
    return deck


def draft_slide(deck: Deck, instruction: str) -> None:
    i = deck.current
    s = deck.slides[i]
    done = [f"{k + 1}. {x.title}" for k, x in enumerate(deck.slides[:i])]
    prompt = f"""You help a user build a presentation one slide at a time.
Topic / goal: {deck.topic or '(see outline)'}
Full outline:
{numbered(deck.outline)}
Slides already written: {done or 'none yet'}

Current slide {i + 1} of {len(deck.slides)}, outline line: "{s.intent}"
Current content (may be empty or contain the user's own edits - keep their wording
unless they ask you to change it):
{s.model_dump_json(include={'title', 'bullets', 'layout', 'notes'})}

User request: {instruction}

Rules:
- title: short and specific
- bullets: 3-5 items, max ~12 words each
- layout: one of {LAYOUTS}
  - "title": one bullet used as the subtitle
  - "quote": bullets[0] is the quote, bullets[1] the attribution
  - "two_column": bullets are split across two columns
  - "image_left": bullets go on the right, the left side is reserved for an image
- notes: 1-3 sentences of speaker notes
Return the updated slide as JSON."""
    new = llm(prompt, SlideDraft)
    s.title, s.bullets, s.layout, s.notes = new.title, new.bullets, new.layout, new.notes
    if s.status != "approved":
        s.status = "draft"
    s.chat.append({"role": "assistant", "content": f"Updated slide: **{new.title}**"})


def draft_current(deck: Deck):
    """Auto-draft the current slide the first time it's visited."""
    if deck.slides and deck.slides[deck.current].status == "pending":
        draft_slide(deck, "Write a first draft of this slide.")
        save(deck)
    return render(deck)


def slide_chat(msg, deck: Deck, title, bullets, layout, notes):
    deck = read_fields(deck, title, bullets, layout, notes)
    msg = (msg or "").strip()
    if msg:
        deck.slides[deck.current].chat.append({"role": "user", "content": msg})
        draft_slide(deck, msg)
    save(deck)
    return render(deck)


def regenerate(deck: Deck, title, bullets, layout, notes):
    deck = read_fields(deck, title, bullets, layout, notes)
    deck.slides[deck.current].chat.append({"role": "user", "content": "(regenerate)"})
    draft_slide(deck, "Regenerate this slide with a fresh take. Ignore the current wording.")
    save(deck)
    return render(deck)


def approve_next(deck: Deck, title, bullets, layout, notes):
    deck = read_fields(deck, title, bullets, layout, notes)
    deck.slides[deck.current].status = "approved"
    if deck.current < len(deck.slides) - 1:
        deck.current += 1
        save(deck)
        return render(deck)
    save(deck)
    gr.Info("All slides approved. Head to the Export tab.")
    return render(deck, "export")


def go_back(deck: Deck, title, bullets, layout, notes):
    deck = read_fields(deck, title, bullets, layout, notes)
    deck.current = max(0, deck.current - 1)
    save(deck)
    return render(deck)


def resume():
    if not SAVE_PATH.exists():
        raise gr.Error("No saved deck found.")
    deck = Deck.model_validate_json(SAVE_PATH.read_text(encoding="utf-8"))
    sel = "slides" if deck.outline_locked else "outline"
    return (*render(deck, sel), "\n".join(deck.outline), deck.outline_chat)


# ----------------------------------------------------------------------------
# Export
# ----------------------------------------------------------------------------
def fill(tf, items):
    tf.text = items[0] if items else ""
    for it in items[1:]:
        tf.add_paragraph().text = it


def build_pptx(deck: Deck):
    prs = Presentation()  # default 4:3 template (10in x 7.5in)
    lay = prs.slide_layouts
    for s in deck.slides:
        b = s.bullets
        if s.layout == "title":
            sl = prs.slides.add_slide(lay[0])
            sl.shapes.title.text = s.title
            sl.placeholders[1].text = "\n".join(b)
        elif s.layout == "two_column":
            sl = prs.slides.add_slide(lay[3])
            sl.shapes.title.text = s.title
            half = math.ceil(len(b) / 2)
            fill(sl.placeholders[1].text_frame, b[:half])
            fill(sl.placeholders[2].text_frame, b[half:])
        elif s.layout == "quote":
            sl = prs.slides.add_slide(lay[5])
            sl.shapes.title.text = s.title
            tb = sl.shapes.add_textbox(Inches(1), Inches(2), Inches(8), Inches(3.5))
            tf = tb.text_frame
            tf.word_wrap = True
            tf.text = b[0] if b else ""
            tf.paragraphs[0].font.size = Pt(28)
            tf.paragraphs[0].font.italic = True
            for extra in b[1:]:
                p = tf.add_paragraph()
                p.text = extra
                p.font.size = Pt(18)
        elif s.layout == "image_left":
            sl = prs.slides.add_slide(lay[5])
            sl.shapes.title.text = s.title
            box = sl.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0.5), Inches(2), Inches(4), Inches(4))
            box.text_frame.text = "Image"
            tb = sl.shapes.add_textbox(Inches(5), Inches(2), Inches(4.5), Inches(4))
            tb.text_frame.word_wrap = True
            fill(tb.text_frame, b)
            for p in tb.text_frame.paragraphs:
                p.font.size = Pt(20)
        else:  # bullets
            sl = prs.slides.add_slide(lay[1])
            sl.shapes.title.text = s.title
            fill(sl.placeholders[1].text_frame, b)
        if s.notes:
            sl.notes_slide.notes_text_frame.text = s.notes
    prs.save(EXPORT_PATH)
    return str(EXPORT_PATH)


def export(deck: Deck):
    if not deck.slides:
        raise gr.Error("Nothing to export yet.")
    return build_pptx(deck)


# ----------------------------------------------------------------------------
# UI
# ----------------------------------------------------------------------------
def make_chatbot(**kw):
    try:
        return gr.Chatbot(type="messages", **kw)  # Gradio 5
    except TypeError:
        return gr.Chatbot(**kw)  # Gradio 6+ (messages format is the only one)


with gr.Blocks(title="Presentation Creator") as demo:
    deck_state = gr.State(Deck())
    gr.Markdown("# Presentation Creator")
    resume_btn = gr.Button("Resume last session (deck.json)", size="sm")

    with gr.Tabs() as tabs:
        with gr.Tab("1. Outline", id="outline"):
            with gr.Row():
                with gr.Column():
                    chat1 = make_chatbot(height=380, label="Outline chat")
                    msg1 = gr.Textbox(
                        placeholder="Describe the presentation, then refine: 'add a slide on pricing', 'merge 3 and 4'...",
                        show_label=False)
                with gr.Column():
                    outline_box = gr.Textbox(
                        label="Outline (one slide per line, edit freely)", lines=16)
                    approve_outline_btn = gr.Button("Approve outline and start slides", variant="primary")

        with gr.Tab("2. Slides", id="slides", interactive=False) as slides_tab:
            progress = gr.Markdown()
            with gr.Row():
                with gr.Column():
                    chat2 = make_chatbot(height=380, label="Slide chat")
                    msg2 = gr.Textbox(placeholder="e.g. make it shorter, add a statistic, switch to two columns",
                                      show_label=False)
                with gr.Column():
                    title = gr.Textbox(label="Title")
                    bullets = gr.Textbox(label="Bullets (one per line)", lines=6)
                    layout = gr.Dropdown(LAYOUTS, label="Layout", value="bullets")
                    notes = gr.Textbox(label="Speaker notes", lines=3)
                    with gr.Row():
                        back_btn = gr.Button("Back")
                        regen_btn = gr.Button("Regenerate")
                        approve_btn = gr.Button("Approve and next", variant="primary")

        with gr.Tab("3. Export", id="export", interactive=False) as export_tab:
            export_btn = gr.Button("Build .pptx", variant="primary")
            file_out = gr.File(label="Your presentation")

    # Every slide-phase handler returns this same set of outputs via render()
    OUT = [deck_state, tabs, slides_tab, export_tab, progress,
           title, bullets, layout, notes, chat2, msg2]
    FORM = [deck_state, title, bullets, layout, notes]

    msg1.submit(outline_chat, [msg1, deck_state, outline_box],
                [deck_state, chat1, msg1, outline_box])

    approve_outline_btn.click(approve_outline, [deck_state, outline_box], OUT) \
        .success(draft_current, deck_state, OUT)

    msg2.submit(slide_chat, [msg2, *FORM], OUT)
    regen_btn.click(regenerate, FORM, OUT)
    back_btn.click(go_back, FORM, OUT)
    approve_btn.click(approve_next, FORM, OUT).success(draft_current, deck_state, OUT)

    export_btn.click(export, deck_state, file_out)
    resume_btn.click(resume, None, [*OUT, outline_box, chat1])

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "C:\Users\User\AppData\Local\Temp\ipykernel_16048\3155034294.py", line 78, in llm
    return model_cls.model_validate_json(r.message.content)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\Documents\GitHub\deckmaker\venv\Lib\site-packages\pydantic\main.py", line 845, in model_validate_json
    return cls.__pydantic_validator__.validate_json(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        json_data, strict=strict, extra=extra, context=context, by_alias=by_alias, by_name=by_name
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
pydantic_core._pydantic_core.ValidationError: 1 validation error for SlideDraft
  Invalid JSON: EOF while parsing a value at line 1 column 0 [type=json_invalid, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/json_invalid

During handling of the above exception, 